# Medical Insurance Cost Analysis

**CISC7204 Data Science Portfolio Project**

## Research Question

What demographic and lifestyle factors drive individual medical insurance charges, and how accurately can a linear regression model predict those charges from the available variables?

This notebook takes a public insurance dataset from raw CSV to cleaned data, exploratory analysis, visualizations, and a reproducible predictive model.


## 1. Setup

We use Pandas and NumPy for data handling, Matplotlib and Seaborn for visualization, and Scikit-learn for the regression model.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.2f}".format)

## 2. Load the Data

In [2]:
df = pd.read_csv("insurance.csv")
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.90,0,yes,southwest,16884.92
1,18,male,33.77,1,no,southeast,1725.55
2,28,male,33.00,3,no,southeast,4449.46
3,33,male,22.70,0,no,northwest,21984.47
4,32,male,28.88,0,no,northwest,3866.86


## 3. Data Overview

The dataset contains 1,338 individuals with seven columns:

- `age`: age of the primary beneficiary
- `sex`: insurance contractor gender
- `bmi`: body mass index
- `children`: number of children covered by the insurance
- `smoker`: smoking status
- `region`: residential area in the US
- `charges`: individual medical costs billed by the insurance (target variable)

In [3]:
print("Shape:", df.shape)
df.info()

Shape: (1338, 7)
<class 'pandas.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   str    
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   str    
 5   region    1338 non-null   str    
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), str(3)
memory usage: 73.3 KB


In [4]:
df.describe(include="all")

,age,sex,bmi,children,smoker,region,charges
count,1338.00,1338,1338.00,1338.00,1338,1338,1338.00
unique,NaN,2,NaN,NaN,2,4,NaN
top,NaN,male,NaN,NaN,no,southeast,NaN
freq,NaN,676,NaN,NaN,1064,364,NaN
mean,39.21,NaN,30.66,1.09,NaN,NaN,13270.42
std,14.05,NaN,6.10,1.21,NaN,NaN,12110.01
min,18.00,NaN,15.96,0.00,NaN,NaN,1121.87
25%,27.00,NaN,26.30,0.00,NaN,NaN,4740.29
50%,39.00,NaN,30.40,1.00,NaN,NaN,9382.03
75%,51.00,NaN,34.69,2.00,NaN,NaN,16639.91


## 4. Data Wrangling

In [5]:
# Check for missing values
df.isna().sum()

age         0
sex         0
bmi         0
children    0
smoker      0
region      0
charges     0
dtype: int64

In [6]:
# Check for duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Remove the single duplicate record
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

Duplicate rows: 1
Shape after removing duplicates: (1337, 7)


In [7]:
# Encode binary categorical variables for modelling
df_encoded = df.copy()
df_encoded["sex"] = df_encoded["sex"].map({"female": 0, "male": 1})
df_encoded["smoker"] = df_encoded["smoker"].map({"no": 0, "yes": 1})

# One-hot encode the region variable
df_encoded = pd.get_dummies(df_encoded, columns=["region"], drop_first=True)
df_encoded.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.90,0,1,16884.92,False,False,True
1,18,1,33.77,1,0,1725.55,False,True,False
2,28,1,33.00,3,0,4449.46,False,True,False
3,33,1,22.70,0,0,21984.47,True,False,False
4,32,1,28.88,0,0,3866.86,True,False,False


## 5. Exploratory Data Analysis

### 5.1 Charges by Smoking Status

The first visualization compares the full distribution of charges for smokers and non-smokers.

In [8]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.histplot(data=df, x="charges", hue="smoker", bins=40, kde=True, ax=ax)
ax.set_title("Distribution of Medical Insurance Charges by Smoking Status")
ax.set_xlabel("Charges (USD per year)")
ax.set_ylabel("Number of Individuals")
plt.tight_layout()
plt.savefig("visualizations/fig1_charges_by_smoker.png", dpi=150)
plt.show()

/tmp/ipykernel_1839/524037516.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [9]:
df.groupby("smoker")["charges"].agg(["mean", "median", "count"]).round(2)

,mean,median,count
smoker,,,
no,8440.66,7345.73,1063
yes,32050.23,34456.35,274


Smokers are charged roughly **four times more** on average than non-smokers, and almost no non-smoker reaches the highest charge bands.

### 5.2 Age versus Charges

In [10]:
fig, ax = plt.subplots(figsize=(9, 5))
sns.scatterplot(data=df, x="age", y="charges", hue="smoker",
                style="smoker", alpha=0.7, ax=ax)
ax.set_title("Age versus Medical Insurance Charges")
ax.set_xlabel("Age")
ax.set_ylabel("Charges (USD per year)")
plt.tight_layout()
plt.savefig("visualizations/fig2_age_vs_charges.png", dpi=150)
plt.show()

/tmp/ipykernel_1839/828450968.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Charges rise with age in three parallel bands, and the upper band is made up almost entirely of smokers.

### 5.3 Charges by Region

In [11]:
fig, ax = plt.subplots(figsize=(9, 5))
region_order = df.groupby("region")["charges"].mean().sort_values(ascending=False).index
sns.boxplot(data=df, x="region", y="charges", order=region_order, ax=ax)
ax.set_title("Medical Insurance Charges by Region")
ax.set_xlabel("Region")
ax.set_ylabel("Charges (USD per year)")
plt.tight_layout()
plt.savefig("visualizations/fig3_charges_by_region.png", dpi=150)
plt.show()

/tmp/ipykernel_1839/118583520.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
df.groupby("region")["charges"].mean().round(2).sort_values(ascending=False)

region
southeast   14735.41
northeast   13406.38
northwest   12450.84
southwest   12346.94
Name: charges, dtype: float64

Regional differences are small compared with the smoker effect, although the southeast has slightly higher average charges.

### 5.4 Correlation with Charges

In [13]:
correlations = (
    df_encoded.corr(numeric_only=True)["charges"]
    .sort_values(ascending=False)
    .round(3)
)
correlations

charges             1.00
smoker              0.79
age                 0.30
bmi                 0.20
region_southeast    0.07
children            0.07
sex                 0.06
region_northwest   -0.04
region_southwest   -0.04
Name: charges, dtype: float64

Smoking status has by far the strongest correlation with charges, followed by age and BMI.

## 6. Predictive Model

We fit a multiple linear regression model on 80% of the data and evaluate it on the held-out 20%.

In [14]:
X = df_encoded.drop(columns="charges")
y = df_encoded["charges"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("R-squared: ", round(r2_score(y_test, y_pred), 3))
print("MAE:       ", round(mean_absolute_error(y_test, y_pred), 2))
print("RMSE:      ", round(np.sqrt(mean_squared_error(y_test, y_pred)), 2))

R-squared:  0.807
MAE:        4177.05
RMSE:       5956.34


In [15]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(y_test, y_pred, alpha=0.6)
max_value = max(y_test.max(), y_pred.max())
ax.plot([0, max_value], [0, max_value], "r--", linewidth=1.5)
ax.set_title("Predicted versus Actual Charges")
ax.set_xlabel("Actual Charges (USD)")
ax.set_ylabel("Predicted Charges (USD)")
plt.tight_layout()
plt.savefig("visualizations/fig4_predicted_vs_actual.png", dpi=150)
plt.show()

/tmp/ipykernel_1839/1162213394.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Points above the dashed line are under-predictions; the model captures the general trend but struggles with the banded, high-charge smoker records.

In [16]:
coefficients = pd.Series(model.coef_, index=X.columns).sort_values(
    key=lambda s: s.abs(), ascending=False
).round(2)
coefficients

smoker             23077.76
region_southeast    -838.92
region_southwest    -659.14
children             533.01
region_northwest    -391.76
bmi                  318.70
age                  248.21
sex                 -101.54
dtype: float64

Holding other variables constant, smoking adds the largest premium (about USD 23,000 per year), while BMI and age add smaller positive amounts.

## 7. Conclusions

### Key Findings

1. **Smoking is the dominant driver of charges**: smokers are charged about four times more on average (around USD 32,000 versus USD 8,400 for non-smokers), and smoking has the strongest correlation and the largest regression coefficient.
2. **Age and BMI matter**: charges increase steadily with age and with BMI, and the age-versus-charges plot reveals three parallel bands.
3. **Regional and gender effects are small** compared with smoking, age, and BMI.
4. A multiple linear regression model explains about **81% of the variation** in charges, with an average absolute error of roughly USD 4,200.

### Limitation

The model assumes a simple linear relationship and cannot reproduce the banded, nonlinear structure visible in the scatter plots; it also relies on a single year of US data with no information about pre-existing medical conditions, so the results should not be generalized to other populations or time periods without validation.

### Next Steps

- Fit a tree-based model (gradient boosting) to capture the nonlinear bands.
- Add interaction terms such as `smoker × age` and `smoker × bmi`.
- Validate the model with cross-validation and compare models on the same splits.
